In [0]:
dbutils.widgets.text("purchase_month", "2018-01", "Purchase month (YYYY-MM)")

In [0]:
%sql
WITH batch AS (
  SELECT CAST(CONCAT(:purchase_month, '-01') AS DATE) AS batch_start
)
INSERT INTO olist.bronze.orders
REPLACE WHERE purchase_month = CAST(CONCAT(:purchase_month, '-01') AS DATE)
SELECT source_orders.*, batch.batch_start AS purchase_month
FROM read_files(
  '/Volumes/olist/source/raw_files/olist_orders_dataset.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false,
  schemaEvolutionMode => 'none'
) AS source_orders
CROSS JOIN batch
WHERE CAST(source_orders.order_purchase_timestamp AS TIMESTAMP) >= batch.batch_start
  AND CAST(source_orders.order_purchase_timestamp AS TIMESTAMP) < ADD_MONTHS(batch.batch_start, 1)

num_affected_rows,num_inserted_rows
6728,6728
